# Toy Project Demo

This notebook demonstrates the basic usage of **AlgoKit Graph** APIs on a small graph. It serves as a quick introduction to graph creation, algorithm execution, and interpreting outputs before moving to large-scale benchmarks.

---

# **Installing AlgoKit Graph v1.0.0**

In [ ]:
!pip install git+https://github.com/ayush09062004/algokit-graph.git

  Cloning https://github.com/ayush09062004/algokit-graph.git to /tmp/pip-req-build-55triu7j
  Running command git clone --filter=blob:none --quiet https://github.com/ayush09062004/algokit-graph.git /tmp/pip-req-build-55triu7j
  Resolved https://github.com/ayush09062004/algokit-graph.git to commit 8a482ae9ce6cf04530c5d4b1776d3793d886b680
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for algokit-graph: filename=algokit_graph-1.0.0-cp312-cp312-linux_x86_64.whl size=161307 sha256=e80ae69f75aa317db11b8c5d99f64f963c1527013fd7ae64988277206000c88c
  Stored in directory: /tmp/pip-ephem-wheel-cache-db0oxbos/wheels/c9/11/96/1c5e8cb247e61e67373cd3f8ec7019932d7e70a83262a6d4e6
Successfully built algokit-graph


# **Create a toy social network**

In [ ]:
import pandas as pd

edges = pd.DataFrame({
    "source": [
        "Alice","Alice","Bob","Carol",
        "David","Eve","Frank","Grace",
        "Heidi","Ivan","Judy","Mallory"
    ],
    "target": [
        "Bob","Carol","David","Alice",
        "Eve","Frank","Grace","Heidi",
        "Ivan","Judy","Mallory","Heidi"
    ]
})

edges.to_csv("friends.csv", index=False)

edges

,source,target
0,Alice,Bob
1,Alice,Carol
2,Bob,David
3,Carol,Alice
4,David,Eve
5,Eve,Frank
6,Frank,Grace
7,Grace,Heidi
8,Heidi,Ivan
9,Ivan,Judy


Read the CSV

In [ ]:
df = pd.read_csv("friends.csv")

df.head()

,source,target
0,Alice,Bob
1,Alice,Carol
2,Bob,David
3,Carol,Alice
4,David,Eve


In [ ]:
people = sorted(
    set(df.source).union(df.target)
)

name_to_id = {
    name:i
    for i,name in enumerate(people)
}

id_to_name = {
    i:name
    for name,i in name_to_id.items()
}

print(name_to_id)

{'Alice': 0, 'Bob': 1, 'Carol': 2, 'David': 3, 'Eve': 4, 'Frank': 5, 'Grace': 6, 'Heidi': 7, 'Ivan': 8, 'Judy': 9, 'Mallory': 10}


# **Building Graph(One edge at a time)**

In [ ]:
import algokit

g1 = algokit.Graph.directed(len(people))

for _, row in df.iterrows():

    g1.add_edge(
        name_to_id[row.source],
        name_to_id[row.target]
    )

print(g1)

Graph(num_vertices=11, num_edges=12, directed=True)


In [ ]:
print("Users :", g1.vertex_count())
print("Edges :", g1.edge_count())

Users : 11
Edges : 12


In [ ]:
scc = g1.strongly_connected_components()

print("Strongly Connected Groups:")

for component in scc.components():

    print(
        [id_to_name[v] for v in component]
    )

Strongly Connected Groups:
['Alice', 'Carol']
['Bob']
['David']
['Eve']
['Frank']
['Grace']
['Heidi', 'Mallory', 'Judy', 'Ivan']


In [ ]:
source = "Alice"

bfs = g1.bfs(
    name_to_id[source]
)

print("Reachable from Alice:")

for v in bfs.order():

    print(id_to_name[v])

Reachable from Alice:
Alice
Bob
Carol
David
Eve
Frank
Grace
Heidi
Ivan
Judy
Mallory


In [ ]:
sp = g1.dijkstra(
    name_to_id["Alice"]
)

target = "Grace"

path = sp.path_to(
    name_to_id[target]
)

print(
    " -> ".join(
        id_to_name[v]
        for v in path
    )
)

Alice -> Bob -> David -> Eve -> Frank -> Grace


In [ ]:
print(
    "Contains cycle:",
    g1.has_directed_cycle()
)

Contains cycle: True


# **Bulk Creation of Edges**(Recommended)

In [ ]:
import pandas as pd
import algokit

# Load CSV
df = pd.read_csv("friends.csv")

# Map names -> vertex IDs
people = sorted(set(df.source).union(df.target))

name_to_id = {name: i for i, name in enumerate(people)}
id_to_name = {i: name for name, i in name_to_id.items()}

# Build edge list
edges = [
    (name_to_id[src], name_to_id[dst])
    for src, dst in zip(df.source, df.target)
]

# Build graph in one call
g = algokit.Graph.directed(len(people))
g.add_edges(edges)

In [ ]:
g

Graph(num_vertices=11, num_edges=12, directed=True)

In [ ]:
import time

# Build edge list (do not include this in graph insertion timing)
edges = [
    (name_to_id[src], name_to_id[dst])
    for src, dst in zip(df.source, df.target)
]

RUNS = 1000

start = time.perf_counter()

for _ in range(RUNS):
    g = algokit.Graph.directed(len(people))
    g.add_edges(edges)

bulk_time = (time.perf_counter() - start) / RUNS

In [ ]:
start = time.perf_counter()

for _ in range(RUNS):
    g = algokit.Graph.directed(len(people))

    for _, row in df.iterrows():
        g.add_edge(
            name_to_id[row.source],
            name_to_id[row.target]
        )

single_time = (time.perf_counter() - start) / RUNS

In [ ]:
print(f"Bulk add_edges : {bulk_time:.8f} s")
print(f"Repeated add_edge : {single_time:.8f} s")
print(f"Speedup : {single_time / bulk_time:.2f}x")

Bulk add_edges : 0.00000345 s
Repeated add_edge : 0.00085376 s
Speedup : 247.71x


In [ ]:
import random
import pandas as pd

N = 100000
M = 500000

edges_df = pd.DataFrame({
    "source": [random.randrange(N) for _ in range(M)],
    "target": [random.randrange(N) for _ in range(M)]
})

people = list(range(N))
name_to_id = {i: i for i in people}

edges = list(zip(edges_df.source, edges_df.target))

In [ ]:
import time
import gc

RUNS = 20

gc.collect()

start = time.perf_counter()

for _ in range(RUNS):

    g = algokit.Graph.directed(N)

    g.add_edges(edges)

bulk_time = (time.perf_counter() - start) / RUNS

In [ ]:
gc.collect()

start = time.perf_counter()

for _ in range(RUNS):

    g = algokit.Graph.directed(N)

    for src, dst in zip(edges_df.source, edges_df.target):

        g.add_edge(
            src,
            dst
        )

single_time = (time.perf_counter() - start) / RUNS

In [ ]:
print("=" * 40)

print(f"Bulk add_edges() : {bulk_time:.6f} s")

print(f"Repeated add_edge(): {single_time:.6f} s")

print(f"Speedup : {single_time / bulk_time:.2f}x")

print("=" * 40)

Bulk add_edges() : 0.219826 s
Repeated add_edge(): 0.561310 s
Speedup : 2.55x
